In [1]:
import pandas as pd
import numpy as np
import math
import os
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from collections import Counter

c:\Users\miku3\OneDrive\Dokumen\Skripsi\Perhitungan-tfidf\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Tahap 1: Load Data dari kaggle
# Download dataset
path = kagglehub.dataset_download('farhan999/tokopedia-product-reviews')
print(f'\nDirektori Dataset: {path}\n')

# Setup display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Load CSV
csv_file_path = os.path.join(path, 'tokopedia-product-reviews-2019.csv')
if os.path.exists(csv_file_path):
    df = pd.read_csv(csv_file_path)
    print(f"Berhasil memuat dataset. Total baris: {len(df)}\n")
else:
    print(f"Error: File tidak ditemukan di: {csv_file_path}")
    exit()


Direktori Dataset: C:\Users\miku3\.cache\kagglehub\datasets\farhan999\tokopedia-product-reviews\versions\2

Berhasil memuat dataset. Total baris: 40607



In [3]:
# Tahap 2: Mengambil sampel untuk simulasi (5 label ordinal)
df_simulasi = df.copy()
df_simulasi['num_words'] =  df_simulasi['text'].apply(lambda x: len(str(x).split()))

# Ambil 1 sampel dari masing2 rating
sampel_data = []
for rating in [5, 4, 3, 2, 1]:
    sampel = df_simulasi[(df_simulasi['rating'] == rating) &
                         (df_simulasi['num_words'] >= 3) &
                         (df_simulasi['num_words']<= 7)].iloc[0:1]
    if not sampel.empty:
        sampel_data.append(sampel)

df_sampel = pd.concat(sampel_data, ignore_index=True)

print("\n=== Sampel Data Latih (5 Label) ===")
for idx, row in df_sampel.iterrows():
    print(f"D{idx+1} (rating {row['rating']}): {row['text']}")


=== Sampel Data Latih (5 Label) ===
D1 (rating 5): Barang sesuai pesanan dan cepat sampai
D2 (rating 4): Barang bagus sesuai info fast response
D3 (rating 3): Barang bagus, kualitas sesuai harga
D4 (rating 2): Pesanan gx sesuai dgn yg d gambar
D5 (rating 1): barang yg dikirim tidak sesuai pesanan


In [4]:
# Tahap 3: Preprocessing & Tokenisasi (Simpel)
def preprocess(text):
    # case folding + tokenisasi
    return text.lower().split()

docs_tokens = [preprocess(doc) for doc in df_sampel['text']]
print("\n=== Hasil Tokenisasi ===")
for i, tokens in enumerate(docs_tokens, 1):
    print(f"D{i}: {tokens}")


=== Hasil Tokenisasi ===
D1: ['barang', 'sesuai', 'pesanan', 'dan', 'cepat', 'sampai']
D2: ['barang', 'bagus', 'sesuai', 'info', 'fast', 'response']
D3: ['barang', 'bagus,', 'kualitas', 'sesuai', 'harga']
D4: ['pesanan', 'gx', 'sesuai', 'dgn', 'yg', 'd', 'gambar']
D5: ['barang', 'yg', 'dikirim', 'tidak', 'sesuai', 'pesanan']


In [5]:
# Tahap 4: Hitung DF, IDF, TF, dan IDF

# A. Hitung DF (Document Frequency)
all_terms = set()
for tokens in docs_tokens:
    all_terms.update(tokens)

df_dict = {}
for term in all_terms:
    df_dict[term] = sum(1 for tokens in docs_tokens if term in tokens)

print("\n=== Document Frequncy (df) ===")
for term in sorted(all_terms):
    docs_dengan_term = [f"D{i+1}" for i, tokens in enumerate(docs_tokens) if term in tokens]
    print(f"{term:15} -> muncul di {docs_dengan_term}, df = {df_dict[term]}")


=== Document Frequncy (df) ===
bagus           -> muncul di ['D2'], df = 1
bagus,          -> muncul di ['D3'], df = 1
barang          -> muncul di ['D1', 'D2', 'D3', 'D5'], df = 4
cepat           -> muncul di ['D1'], df = 1
d               -> muncul di ['D4'], df = 1
dan             -> muncul di ['D1'], df = 1
dgn             -> muncul di ['D4'], df = 1
dikirim         -> muncul di ['D5'], df = 1
fast            -> muncul di ['D2'], df = 1
gambar          -> muncul di ['D4'], df = 1
gx              -> muncul di ['D4'], df = 1
harga           -> muncul di ['D3'], df = 1
info            -> muncul di ['D2'], df = 1
kualitas        -> muncul di ['D3'], df = 1
pesanan         -> muncul di ['D1', 'D4', 'D5'], df = 3
response        -> muncul di ['D2'], df = 1
sampai          -> muncul di ['D1'], df = 1
sesuai          -> muncul di ['D1', 'D2', 'D3', 'D4', 'D5'], df = 5
tidak           -> muncul di ['D5'], df = 1
yg              -> muncul di ['D4', 'D5'], df = 2


In [6]:
# B. Hitung IDF (Inverse Document Frequency)
N = len(df_sampel)
idf_dict = {}
for term in all_terms:
    idf_dict[term] = math.log10(N /df_dict[term])

print(f"\n=== Inverse Document Frequency (idf = log₁₀(N/df)) ===")
print(f"N (total dokumen) = {N}\n")
for term in sorted(all_terms):
    print(f"{term:15} -> log₁₀({N}/{df_dict[term]}) = {idf_dict[term]:.3f}")


=== Inverse Document Frequency (idf = log₁₀(N/df)) ===
N (total dokumen) = 5

bagus           -> log₁₀(5/1) = 0.699
bagus,          -> log₁₀(5/1) = 0.699
barang          -> log₁₀(5/4) = 0.097
cepat           -> log₁₀(5/1) = 0.699
d               -> log₁₀(5/1) = 0.699
dan             -> log₁₀(5/1) = 0.699
dgn             -> log₁₀(5/1) = 0.699
dikirim         -> log₁₀(5/1) = 0.699
fast            -> log₁₀(5/1) = 0.699
gambar          -> log₁₀(5/1) = 0.699
gx              -> log₁₀(5/1) = 0.699
harga           -> log₁₀(5/1) = 0.699
info            -> log₁₀(5/1) = 0.699
kualitas        -> log₁₀(5/1) = 0.699
pesanan         -> log₁₀(5/3) = 0.222
response        -> log₁₀(5/1) = 0.699
sampai          -> log₁₀(5/1) = 0.699
sesuai          -> log₁₀(5/5) = 0.000
tidak           -> log₁₀(5/1) = 0.699
yg              -> log₁₀(5/2) = 0.398


In [7]:
# C. Hitung TF (Term Frequency)
tf_dict = {}
for doc_id, tokens in enumerate(docs_tokens):
    doc_name = f'D{doc_id + 1}'
    tf_dict[doc_name] = {}
    for term in all_terms:
        tf_dict[doc_name][term] = tokens.count(term)

print(f"\n=== Term Frequency (tf) ===")
for doc_name in sorted(tf_dict.keys()):
    print(f"\n{doc_name}:")
    for term in sorted(tf_dict[doc_name].keys()):
        if tf_dict[doc_name][term] > 0:
            print(f" {term:5} = {tf_dict[doc_name][term]}")


=== Term Frequency (tf) ===

D1:
 barang = 1
 cepat = 1
 dan   = 1
 pesanan = 1
 sampai = 1
 sesuai = 1

D2:
 bagus = 1
 barang = 1
 fast  = 1
 info  = 1
 response = 1
 sesuai = 1

D3:
 bagus, = 1
 barang = 1
 harga = 1
 kualitas = 1
 sesuai = 1

D4:
 d     = 1
 dgn   = 1
 gambar = 1
 gx    = 1
 pesanan = 1
 sesuai = 1
 yg    = 1

D5:
 barang = 1
 dikirim = 1
 pesanan = 1
 sesuai = 1
 tidak = 1
 yg    = 1


In [8]:
# D. Hitung TF-IDF
tfidf_dict = {}
for doc_id, tokens in enumerate(docs_tokens):
    doc_name = f'D{doc_id + 1}'
    tfidf_dict[doc_name] = {}
    for term in all_terms:
        tfidf_dict[doc_name][term] = tf_dict[doc_name][term] * idf_dict[term]

In [12]:
# Tahap 5: Buat Tabel Hasil
print("TABEL HASIL: DF, IDF, TF, dan TF-IDF\n")

tfidf_results = []
for term in sorted(all_terms):
    row = {
        'Term (t)': term,
        'df': df_dict[term],
        'idf': f"{idf_dict[term]:.3f}",
        'TF(D₁)': tf_dict['D1'][term],
        'TF(D₂)': tf_dict['D2'][term],
        'TF(D₃)': tf_dict['D3'][term],
        'TF(D₄)': tf_dict['D4'][term],
        'TF(D₅)': tf_dict['D5'][term],
        'TF-IDF(D₁)': f"{tfidf_dict['D1'][term]:.3f}",
        'TF-IDF(D₂)': f"{tfidf_dict['D2'][term]:.3f}",
        'TF-IDF(D₃)': f"{tfidf_dict['D3'][term]:.3f}",
        'TF-IDF(D₄)': f"{tfidf_dict['D4'][term]:.3f}",
        'TF-IDF(D₅)': f"{tfidf_dict['D5'][term]:.3f}"
    }
    tfidf_results.append(row)

df_hasil = pd.DataFrame(tfidf_results)
print(df_hasil.to_string(index=False))

# Save ke CSV
df_hasil.to_csv('hasil_simulasi_tfidf.csv', index=False)
print("\n Hasil disimpan ke 'hasil_simulasi_tfidf.csv'")

TABEL HASIL: DF, IDF, TF, dan TF-IDF

Term (t)  df   idf  TF(D₁)  TF(D₂)  TF(D₃)  TF(D₄)  TF(D₅) TF-IDF(D₁) TF-IDF(D₂) TF-IDF(D₃) TF-IDF(D₄) TF-IDF(D₅)
   bagus   1 0.699       0       1       0       0       0      0.000      0.699      0.000      0.000      0.000
  bagus,   1 0.699       0       0       1       0       0      0.000      0.000      0.699      0.000      0.000
  barang   4 0.097       1       1       1       0       1      0.097      0.097      0.097      0.000      0.097
   cepat   1 0.699       1       0       0       0       0      0.699      0.000      0.000      0.000      0.000
       d   1 0.699       0       0       0       1       0      0.000      0.000      0.000      0.699      0.000
     dan   1 0.699       1       0       0       0       0      0.699      0.000      0.000      0.000      0.000
     dgn   1 0.699       0       0       0       1       0      0.000      0.000      0.000      0.699      0.000
 dikirim   1 0.699       0       0       0       0

In [13]:
for term in sorted(all_terms):
    print(f"\n{'='*80}")
    print(f"TERM: '{term}'")
    print(f"{'='*80}")
    
    docs_dengan_term = [f"D{i+1}" for i, tokens in enumerate(docs_tokens) if term in tokens]
    print(f"\n1. DOCUMENT FREQUENCY (df):")
    print(f"   Muncul di dokumen: {', '.join(docs_dengan_term)}")
    print(f"   df = {df_dict[term]}")
    
    print(f"\n2. INVERSE DOCUMENT FREQUENCY (idf):")
    print(f"   idf = log₁₀(N / df)")
    print(f"   idf = log₁₀({N} / {df_dict[term]})")
    print(f"   idf = {idf_dict[term]:.3f}")
    
    print(f"\n3. TERM FREQUENCY & TF-IDF di setiap dokumen:")
    for doc_id in range(N):
        doc_name = f'D{doc_id + 1}'
        tf_val = tf_dict[doc_name][term]
        tfidf_val = tfidf_dict[doc_name][term]
        print(f"   {doc_name}: TF = {tf_val}, TF-IDF = {tf_val} × {idf_dict[term]:.3f} = {tfidf_val:.3f}")


TERM: 'bagus'

1. DOCUMENT FREQUENCY (df):
   Muncul di dokumen: D2
   df = 1

2. INVERSE DOCUMENT FREQUENCY (idf):
   idf = log₁₀(N / df)
   idf = log₁₀(5 / 1)
   idf = 0.699

3. TERM FREQUENCY & TF-IDF di setiap dokumen:
   D1: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D2: TF = 1, TF-IDF = 1 × 0.699 = 0.699
   D3: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D4: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D5: TF = 0, TF-IDF = 0 × 0.699 = 0.000

TERM: 'bagus,'

1. DOCUMENT FREQUENCY (df):
   Muncul di dokumen: D3
   df = 1

2. INVERSE DOCUMENT FREQUENCY (idf):
   idf = log₁₀(N / df)
   idf = log₁₀(5 / 1)
   idf = 0.699

3. TERM FREQUENCY & TF-IDF di setiap dokumen:
   D1: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D2: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D3: TF = 1, TF-IDF = 1 × 0.699 = 0.699
   D4: TF = 0, TF-IDF = 0 × 0.699 = 0.000
   D5: TF = 0, TF-IDF = 0 × 0.699 = 0.000

TERM: 'barang'

1. DOCUMENT FREQUENCY (df):
   Muncul di dokumen: D1, D2, D3, D5
   df = 4

2. INVERSE DOCUMENT FREQUENCY (idf):
 